# H5: checkpoint averaging of the locked RAMP baseline
Uniform weight average of epochs 60/70/80 of `ramp_scratch_l5_s{0,1}`, evaluated with the same fixed-ε AutoAttack protocol (first 1000 test images). About 35 session-minutes on T4 x2. GPU **T4 x2**, Internet **on**, then Save Version → Save & Run All. Result: `ramp_official/ramp_scratch_avg_l5_s{0,1}/eval_autoattack.json` on HF.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b claude/read-hand-off-5m8cj2 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

In [ ]:
stage('install', 'bash scripts/ramp_official.sh install')
stage('data', 'bash scripts/ramp_official.sh data')

In [ ]:
stage('avg_eval', 'bash scripts/ramp_official.sh evalavg2 0 1')